# SOTA SFT Data Prep v2 — Spurgeon Q&A (Notebook D_sota)

Plan: `fine_tuning/notebooks/PLAN_FABLE5_TO_IMPROVE_FN.md`

- Input: `qa_mix_train.jsonl` / `qa_mix_val.jsonl` from `build_qa_mix.py`
- Template: ChatML via tokenizer-native `apply_chat_template`
- **No vocab resize** — pad with existing token only
- Outputs `qa_dataset_train/` + `qa_dataset_val/` via `save_to_disk`


## 1. Config

In [ ]:
import json
from pathlib import Path

# Kaggle: mount qa-mix dataset; local: repo fine_tuning/data/
DATA_ROOT = Path("/kaggle/input/datasets/spurgeon-qa-mix-v1")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("../../data")  # local from notebooks/

TRAIN_JSONL = DATA_ROOT / "qa_mix_train.jsonl"
VAL_JSONL = DATA_ROOT / "qa_mix_val.jsonl"
MANIFEST = DATA_ROOT / "qa_mix_manifest.json"
OUT_TRAIN = Path("/kaggle/working/qa_dataset_train") if Path("/kaggle/working").exists() else Path("../../data/qa_dataset_train")
OUT_VAL = Path("/kaggle/working/qa_dataset_val") if Path("/kaggle/working").exists() else Path("../../data/qa_dataset_val")
MAX_SEQ_LENGTH = 4096

CANONICAL_SYSTEM = 'You are Charles Haddon Spurgeon (1834–1892). Answer using only the information in the provided CONTEXT from your sermons. Stay faithful to the text: do not invent facts, quotes, or citations not supported by the context.\n\nIf the CONTEXT does not contain enough information to answer the question, say so briefly in your own voice—do not speculate or apologize at length.\n\nWhen you draw on a specific sermon passage, cite it inline as [Sermon N] when the header is present in the context.'


## 2. Load tokenizer (stock Qwen3.5 for dev; same family as CPT merge)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3.5-4B-Base"
tokenizer = FastLanguageModel.get_tokenizer(MODEL_NAME)

# F2: never resize vocab
assert len(tokenizer) == tokenizer.vocab_size, "vocab resize detected — abort"
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

for t in ["<|im_start|>", "<|im_end|>", "<|endoftext|>"]:
    ids = tokenizer(t, add_special_tokens=False)["input_ids"]
    print(t, "->", ids, "(atomic)" if len(ids) == 1 else "(NOT ATOMIC)")
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)


## 3. Build ChatML dataset + S1 token audit

In [ ]:
import numpy as np
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def to_chatml_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def build_ds(jsonl_path):
    rows = load_jsonl(jsonl_path)
    texts = []
    for ex in rows:
        msgs = ex["messages"]
        if msgs[0]["content"] != CANONICAL_SYSTEM:
            msgs = [{"role": "system", "content": CANONICAL_SYSTEM}] + msgs[1:]
        texts.append({"text": to_chatml_text(msgs), "messages": msgs})
    return Dataset.from_list(texts)

train_ds = build_ds(TRAIN_JSONL)
val_ds = build_ds(VAL_JSONL)
print("train/val:", len(train_ds), len(val_ds))

lens = [len(tokenizer(x["text"])["input_ids"]) for x in train_ds]
print("S1 p50/p90/p99/max:", np.percentile(lens, [50, 90, 99]).astype(int), max(lens))
over = sum(l > MAX_SEQ_LENGTH for l in lens)
print(f"over {{MAX_SEQ_LENGTH}}:", over, f"({100*over/max(1,len(lens)):.1f}%)")
if over > len(lens) * 0.02:
    print("WARNING: >2% examples exceed MAX_SEQ_LENGTH — trim data or lower k before training")


## 4. Save to disk (consumed by E_sota)

In [ ]:
OUT_TRAIN.mkdir(parents=True, exist_ok=True)
OUT_VAL.mkdir(parents=True, exist_ok=True)
train_ds.save_to_disk(str(OUT_TRAIN))
val_ds.save_to_disk(str(OUT_VAL))
print("Saved", OUT_TRAIN, OUT_VAL)
if MANIFEST.exists():
    print("Manifest:", json.loads(MANIFEST.read_text(encoding="utf-8"))["counts"])
